# CAP4611 — Programming Assignment 1: Exploratory Data Analysis

**Author:** Richard Magiday  
**Course:** CAP4611 — Machine Learning  
**Dataset:** `telco_churn2.csv` — ConnectIQ Telecom Customer Churn  
**Date:** June 2026

---

## Introduction

This notebook performs a full **Exploratory Data Analysis (EDA)** on the ConnectIQ Telecom churn dataset. The goal is to thoroughly understand the data before building any predictive model, by:

- Summarizing the data statistically
- Identifying and handling missing values
- Visualizing distributions of **all** categorical and numerical features
- Analyzing correlations and outliers
- Engineering a new feature from `tenure`
- Forming hypotheses about which features are most predictive of churn

---
## Upload Dataset

Run the cell below to upload `telco_churn2.csv` from your local machine into this Colab session.

In [ ]:
from google.colab import files
uploaded = files.upload()   # select telco_churn2.csv when prompted

---
# A. Basic Setup

## A1 — Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries imported successfully.')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'seaborn : {sns.__version__}')

## A2 — Load the Dataset

In [ ]:
df = pd.read_csv('telco_churn2.csv')

print(f'Number of rows   : {df.shape[0]:,}')
print(f'Number of columns: {df.shape[1]}')

## A3 — Summary Statistics

In [ ]:
df.describe()

### Interpretation of Summary Statistics

* **`tenure`** — ranges from **0 to 72 months** with mean ~32 and std ~24. The wide spread means customers are at every stage of their subscription lifecycle, from brand-new to 6-year loyalists.
* **`MonthlyCharges`** — ranges from ~\$18 to ~\$119, mean ~\$65, std ~\$30. The large spread is driven by three broad service tiers: phone-only (cheap), DSL (mid), and Fiber optic (expensive).
* **`TotalCharges`** — `describe()` may show `object` dtype here because the column contains blank strings that disguise it as text. Its high std relative to its mean indicates **right skew** — many short-tenure customers with low totals, and a long tail of high-tenure customers with large accumulated bills. This is corrected in Section B.
* **`SeniorCitizen`** — mean ≈ 0.162, confirming only **~16%** of the customer base is a senior citizen. Despite being stored as int (0/1), this is a binary categorical variable.

## A4 — First 5 and Last 5 Rows

In [ ]:
print('=== First 5 rows ===')
display(df.head())

print('\n=== Last 5 rows ===')
display(df.tail())

## A5 — List All Numerical Columns

> **Note:** `TotalCharges` is currently stored as `object` due to blank strings in the raw CSV. After cleaning in Section B it will become `float64` and will be added to the numerical column list.

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f'Numerical columns as-loaded ({len(num_cols)} total):')
for c in num_cols:
    print(f'  - {c}')
print('\nNote: SeniorCitizen is int (0/1) but is conceptually a binary categorical feature.')
print('Note: TotalCharges is currently object — it will become float64 after Section B conversion.')

## A6 — List All Categorical Columns

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical (object) columns ({len(cat_cols)} total):')
for c in cat_cols:
    print(f'  - {c}  (unique values: {df[c].nunique()})')

---
# B. Missing Values Analysis

## B1 — Column-wise Missing Value Count (Descending)

In [ ]:
mv_count = df.isnull().sum().sort_values(ascending=False)
print('Missing value counts (descending — showing all columns):')
print(mv_count)

## B2 — Column-wise Missing Value Percentage (Descending)

In [ ]:
mv_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('Missing value percentages (descending — showing all columns):')
print(mv_pct.round(2))

## B3 — Convert `TotalCharges` Blank Strings to NaN, Then Re-check

In [ ]:
print(f'TotalCharges dtype BEFORE conversion: {df["TotalCharges"].dtype}')
print(f'Blank-string rows detected          : {(df["TotalCharges"].str.strip() == "").sum()}')

# Convert — blank strings become NaN, valid numbers become float64
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f'\nTotalCharges dtype AFTER conversion : {df["TotalCharges"].dtype}')
print('\nRe-check missing values after conversion (only columns with missing values shown):')
mv_after = df.isnull().sum().sort_values(ascending=False)
print(mv_after[mv_after > 0])

print('\n--- Updated Numerical Columns (after TotalCharges conversion) ---')
num_cols_updated = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f'Numerical columns now ({len(num_cols_updated)} total):')
for c in num_cols_updated:
    print(f'  - {c}')
print('\nTotalCharges is now float64 and part of the numerical feature set.')

## B4 — Bar Plot: Columns with Missing Values (Least → Most)

In [ ]:
mv_plot = df.isnull().sum()
mv_plot = mv_plot[mv_plot > 0].sort_values()   # ascending = least on the left

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(mv_plot.index, mv_plot.values, color='steelblue', edgecolor='black')
ax.set_title('Missing Values per Column (Least → Most)', fontsize=13)
ax.set_xlabel('Column')
ax.set_ylabel('Missing Count')
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 20,
            str(int(b.get_height())), ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## B5a — Missingno Matrix (200-row Sample)

In [ ]:
sample_df = df.sample(200, random_state=42)

fig, ax = plt.subplots(figsize=(14, 6))
msno.matrix(sample_df, ax=ax, sparkline=True)
ax.set_title('Missingno Matrix — 200-row Sample', fontsize=13)
plt.tight_layout()
plt.show()

# Compute the actual sparkline values from the sample
row_completeness = sample_df.notnull().sum(axis=1)
print(f'\n--- Sparkline Values (from the 200-row sample) ---')
print(f'Left-most sparkline value  (minimum row completeness): {row_completeness.min()} non-null columns')
print(f'Right-most sparkline value (maximum row completeness): {row_completeness.max()} non-null columns')
print(f'Total columns in dataset : {df.shape[1]}')

### Sparkline Interpretation

The **spark line** on the right side of the missingno matrix shows **row-level completeness** across the 200-row sample:

* **Left-most value** — the **minimum** number of non-null columns in any single row of the sample (worst-case row completeness). This is the row that is missing the most data.
* **Right-most value** — the **maximum** number of non-null columns found in any single row (best-case row completeness — the most complete rows).

Because only **2 columns** have missing values (`TotalCharges` and `Education`), the sparkline range is very narrow. Most rows are complete or near-complete. The left-most value printed above reflects the worst row in the sample, and the right-most reflects the most complete. A tight sparkline with almost no variance confirms that **missingness is not widespread across rows** in this dataset.

## B5b — Missingno Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
msno.heatmap(df, ax=ax)
ax.set_title('Missingno Heatmap — Co-missingness Correlation', fontsize=13)
plt.tight_layout()
plt.show()

### Why the Missingno Heatmap Is Not Interesting Here

The missingno heatmap measures **co-missingness correlation** — how often two columns are missing *at the same time* across rows. It is only meaningful when **many columns** have missing values, so cross-column missingness patterns can be compared.

In this dataset, only **2 columns** have missing values (`TotalCharges` and `Education`). This produces a nearly empty **2×2 matrix** with essentially one meaningful cell — there is almost nothing to compare. The heatmap is trivially uninformative.

**What the extreme values mean:**
* **+1** — The two columns are *always* missing **together** — whenever one is NaN, the other is also NaN (perfect positive co-missingness).
* **−1** — The two columns *never* miss at the same time — their missingness is mutually exclusive (perfect negative co-missingness).

With only two columns in play, the heatmap doesn't tell us anything actionable about data structure.

## B6 — How to Handle Each Column with Missing Values

After converting `TotalCharges`, two columns have missing values. Below is the resolution plan for each.

### Missing Column List and Solutions

**1. `Education` (~80.6% missing, ~5,679 NaN rows)**
* **Solution:** **Drop the column entirely.**
* **Reasoning:** More than 80% of rows lack a value. Imputing the majority of a feature would manufacture most of its data, introducing noise and bias that could mislead the model. The remaining ~20% of non-null rows (covering only 4 education levels) are far too sparse to be a reliable predictor. We will first plot `Education` in Section C for completeness, then drop it.

**2. `TotalCharges` (~0.16% missing, 11 NaN rows)**
* **Solution:** **Impute with 0.**
* **Reasoning:** All 11 rows with missing `TotalCharges` have `tenure == 0` — brand-new customers who have not yet received a bill. A total charge of \$0 is the logically correct value. Since this affects less than 0.2% of the data and the imputed value is grounded in business logic, this is safe and accurate.

In [ ]:
# Verify that all TotalCharges-missing rows have tenure == 0
missing_tc = df[df['TotalCharges'].isnull()]
print(f'Rows with missing TotalCharges : {len(missing_tc)}')
print('Tenure values for those rows   :')
print(missing_tc[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].to_string())

# Impute TotalCharges with 0 for the tenure-0 rows
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f'\nTotalCharges NaN rows filled with 0.')
print('NOTE: Education will be analyzed in Section C before being dropped.')
print(f'\nRemaining NaN counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])

---
# C. Understanding Categorical Attributes

We analyze **all** categorical attributes, including:
- High-cardinality identifiers (`customerID`, `SupportTicketID`) — handled separately since bar plots are not meaningful for thousands of unique values.
- `Education` — analyzed here before being dropped (>80% missing).
- `ActiveCountryCode` — shown to expose its zero-variance (constant value).
- All remaining categoricals and the target `Churn`.

## C — High-Cardinality Identifiers: `customerID` and `SupportTicketID`

In [ ]:
# These columns have too many unique values for a meaningful bar chart.
# We inspect their cardinality and sample values instead.
for col in ['customerID', 'SupportTicketID']:
    print(f'=== {col} ===')
    print(f'  Total rows  : {len(df)}')
    print(f'  Unique values: {df[col].nunique()}')
    print(f'  Sample values: {df[col].head(5).tolist()}')
    print(f'  Are all values unique? {df[col].nunique() == len(df)}')
    print()

**Why no bar plot for `customerID` and `SupportTicketID`?**

Both columns have **cardinality equal to (or near) the number of rows** — they are unique identifiers. Plotting thousands of bars would produce an unreadable chart and carry zero analytical value. The correct action is to:
* Confirm they are IDs (done above).
* Note they provide **zero predictive value** for churn.
* Plan to **drop both** before modelling.

## C — Bar Plots: Category Counts for All Remaining Categorical Features

Includes `Education` (before dropping) and `ActiveCountryCode` (to expose constant value).

In [ ]:
cat_analysis = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod',
    'Education', 'ActiveCountryCode', 'Churn',
]

n_plot_cols = 2
n_plot_rows = (len(cat_analysis) + 1) // 2

fig, axes = plt.subplots(n_plot_rows, n_plot_cols, figsize=(16, n_plot_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_analysis):
    vc = df[col].value_counts(dropna=False)
    sns.barplot(x=vc.index.astype(str), y=vc.values, ax=axes[i], palette='Set2')
    axes[i].set_title(f'Counts: {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)
    for p in axes[i].patches:
        axes[i].annotate(f'{int(p.get_height())}',
                         (p.get_x() + p.get_width() / 2, p.get_height()),
                         ha='center', va='bottom', fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Category Counts — All Categorical Attributes', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## C — Countplots: Each Feature vs Churn (Target)

In [ ]:
cat_vs_churn = [c for c in cat_analysis if c != 'Churn']

n_plot_rows2 = (len(cat_vs_churn) + 1) // 2
fig, axes = plt.subplots(n_plot_rows2, n_plot_cols, figsize=(16, n_plot_rows2 * 4))
axes = axes.flatten()

for i, col in enumerate(cat_vs_churn):
    sns.countplot(data=df, x=col, hue='Churn', ax=axes[i], palette='Set1')
    axes[i].set_title(f'{col} vs Churn', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(title='Churn', fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Features vs Churn (Target)', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## C — Interpretation and Decisions

### Feature-level Decisions (with Reasoning)

* **`Churn` (target):** ~**26.5% Yes** vs ~**73.5% No** — significant **class imbalance** (~3:1 ratio). A naive classifier that always predicts "No" would achieve 73.5% accuracy without learning anything. **Resampling** (e.g., **SMOTE** oversampling, random undersampling) or **class-weighted** estimators will be required. Prefer **F1-score**, **ROC-AUC**, or Precision-Recall AUC over raw accuracy.

* **`customerID`:** Every row is a unique identifier (7,043 unique values out of 7,043 rows) — **maximum cardinality**, zero predictive signal. → **DROP before modelling.**

* **`SupportTicketID`:** Near-unique ticket reference strings. Zero predictive value and extremely high cardinality. → **DROP before modelling.**

* **`ActiveCountryCode`:** **Zero-variance** — all rows contain the same constant value (`"+1"`). A zero-variance feature adds no information to any model. → **DROP before modelling.**

* **`Education`:** The bar plot shows ~80.6% of values are NaN, with only 4 education levels represented across the remaining ~20% of rows. Imputing over 5,679 rows would manufacture most of the feature. → **DROP** (confirmed from Section B reasoning). Will be dropped immediately after this section.

* **`SeniorCitizen`:** Only ~16% are seniors (1), yet the countplot shows seniors have a noticeably higher churn rate. → **KEEP**; already encoded as binary 0/1.

* **`Contract`:** Month-to-month customers are the largest group **and** churn at the highest rate by far. Two-year contract holders almost never churn. This is expected to be one of the **strongest predictors**. → **KEEP**.

* **`InternetService`:** Fiber optic users churn the most; DSL less so; customers with no internet barely churn. Strong predictive signal. → **KEEP**.

* **`PaymentMethod`:** Electronic check users churn far more than automatic payment users (credit card / bank transfer). → **KEEP**.

* **`OnlineSecurity`, `TechSupport`, `OnlineBackup`, `DeviceProtection`, `StreamingTV`, `StreamingMovies`:** Each has a "No internet service" level directly tied to `InternetService`. These 6 features are **structurally correlated** with `InternetService`. Retain for now, but check **VIF (Variance Inflation Factor)** after one-hot encoding and prune if needed.

* **`gender`:** Roughly 50/50 split; churn rates look nearly identical for both genders. Low predictive power. → **Monitor feature importance** after modelling; strong candidate for dropping.

* **`PaperlessBilling`:** Paperless billing customers churn more. → **KEEP** (informative).

* **`Partner` / `Dependents`:** Customers without partners or dependents churn slightly more. → **KEEP** (modest signal).

### Non-Technical Finding

> Customers who pay by **electronic check** are much more likely to cancel their service than those who use automatic payment methods like credit cards or bank transfers. This likely reflects that customers who never set up autopay are less committed to the relationship — they may be more price-conscious, or more actively comparing ConnectIQ to competitors.

In [ ]:
# Drop columns identified as non-informative after the Section C analysis
cols_to_drop = ['customerID', 'SupportTicketID', 'ActiveCountryCode', 'Education']
df.drop(columns=cols_to_drop, inplace=True)

print('Dropped columns:', cols_to_drop)
print(f'DataFrame shape after drops: {df.shape}')
print(f'Remaining columns: {df.columns.tolist()}')

---
# D. Understanding Numerical Attributes

The three numerical features are `tenure`, `MonthlyCharges`, and `TotalCharges`. (`SeniorCitizen` is binary and analyzed in Section C.)

> **Note on `sns.distplot`:** `sns.distplot()` was **deprecated in seaborn v0.11** and **removed in seaborn v0.12**. Since Colab runs seaborn ≥ 0.12, calling `sns.distplot()` raises an `AttributeError`. The **modern equivalent** is `sns.histplot(kde=True)`, which produces an identical visualization: a histogram combined with a Kernel Density Estimate (KDE) curve. All distplot cells below use this replacement.

## D — Histograms

In [ ]:
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(num_features):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue',
                 edgecolor='black', alpha=0.75)
    axes[i].axvline(df[col].mean(),   color='red',   linestyle='--', linewidth=2,
                    label=f'Mean: {df[col].mean():.1f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', linewidth=2,
                    label=f'Median: {df[col].median():.1f}')
    axes[i].set_title(f'Histogram: {col}', fontsize=13)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=9)

plt.suptitle('Histograms of Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

## D — Seaborn Distplot

`sns.histplot(kde=True)` is used as the modern, non-deprecated replacement for `sns.distplot()`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(num_features):
    # sns.distplot was removed in seaborn 0.12.
    # sns.histplot with kde=True is the direct, documented replacement.
    sns.histplot(df[col].dropna(), kde=True, bins=30,
                 ax=axes[i], color='purple', alpha=0.6)
    axes[i].set_title(f'Distplot (histplot+KDE): {col}', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Density')

plt.suptitle('Seaborn Distplot — histplot + KDE (Modern Equivalent of sns.distplot)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
print('Skewness of numerical features:')
for col in num_features:
    skew = df[col].skew()
    direction = 'right-skewed' if skew > 0.5 else ('left-skewed' if skew < -0.5 else 'roughly symmetric')
    print(f'  {col}: skewness = {skew:.3f}  ({direction})')

## D — Interpretation and Decisions

### Feature-level Decisions

* **`tenure`:** Shows a roughly **bimodal / near-uniform** distribution with spikes at very low values (many new customers) and near the maximum (many long-loyal customers), with a flatter middle region. This bimodal pattern motivates the **tenure_group** categorical feature created in Section G. No strong skew; a **StandardScaler** or **MinMaxScaler** will suffice before distance-based models.

* **`MonthlyCharges`:** Roughly **uniform** with subtle multi-modal peaks (~\$20, ~\$50–70, ~\$90–100). These correspond to the three service tiers: phone-only (cheap), DSL internet (mid), and Fiber optic (expensive). No extreme skew. → Apply **normalization** before models sensitive to feature scale (SVM, KNN, neural networks).

* **`TotalCharges`:** Clearly **right-skewed** (skewness > 1; mean > median). Many short-tenure customers have low total charges, while a small group of long-loyal customers accumulate very large totals. It is also highly collinear with `tenure × MonthlyCharges`. → Consider **log-transformation** (`np.log1p`) to reduce skew if retained. Given its multicollinearity with `tenure` + `MonthlyCharges`, the recommended action is to **DROP `TotalCharges`** (see Section E for confirmation).

### Non-Technical Findings

1. **The bimodal tenure pattern** tells a business story: ConnectIQ has two distinct customer types — those who try the service and leave very quickly (spike at 0–5 months) and those who stick around for years (spike near 72 months). Very few customers stay in the middle range, suggesting there is a critical "commitment threshold" around the 2-year mark — customers who make it past that tend to stay long-term.

2. **Monthly bill price clusters** clearly reflect ConnectIQ's three pricing tiers (~\$20 for basic phone, ~\$50–70 for internet bundles, ~\$90–100 for premium Fiber optic with all add-ons). This multi-modal shape shows there is no single "average" customer — the customer base is naturally segmented by service level.

3. **The right-skewed Total Charges distribution** means the vast majority of customers have paid relatively little to ConnectIQ — they are newer. The long right tail belongs to a small group of long-tenured, loyal customers who have collectively paid the most. Ironically, these high-paying loyalists are also the least likely to churn.

---
# E. Correlation Analysis

## E1 — Correlation Heatmap (Numerical Features + Churn)

In [ ]:
# Create a temporary numeric version of Churn for correlation purposes only
df_corr = df.copy()
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)

corr_cols   = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_num']
corr_labels = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']

corr_matrix = df_corr[corr_cols].corr()
corr_matrix.columns = corr_labels
corr_matrix.index   = corr_labels

print('Correlation Matrix:')
display(corr_matrix.round(3))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, annot_kws={'size': 12}, ax=ax)
ax.set_title('Correlation Heatmap — Numerical Features + Churn', fontsize=13)
plt.tight_layout()
plt.show()

## E2 — Boxplots: Numerical Features vs Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, col in enumerate(num_features):
    sns.boxplot(data=df, x='Churn', y=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} vs Churn', fontsize=13)
    axes[i].set_xlabel('Churn')
    axes[i].set_ylabel(col)

plt.suptitle('Numerical Features vs Churn', fontsize=14)
plt.tight_layout()
plt.show()

# Print median values per Churn group for each feature
print('\nMedian values by Churn group:')
print(df.groupby('Churn')[num_features].median().round(2))

## E — Interpretation and Decisions

### Feature Relationships and Decisions

* **`TotalCharges` vs `tenure` (r ≈ 0.83):** Very high **positive correlation** — this is **multicollinearity**. `TotalCharges` is mathematically near-equivalent to `tenure × MonthlyCharges`. Keeping all three in a linear model produces unstable, inflated coefficients (a symptom of multicollinearity). → **DROP `TotalCharges`** and retain `tenure` + `MonthlyCharges`.

* **`TotalCharges` vs `MonthlyCharges` (r ≈ 0.65):** Moderate positive correlation, further confirming redundancy — higher monthly bills compound into higher totals over time.

* **`Churn` vs `tenure` (r ≈ −0.35):** Moderate **negative correlation** — longer-tenured customers are significantly less likely to churn. **`tenure` is a strong predictor.** The boxplot confirms churned customers have a noticeably lower median tenure.

* **`Churn` vs `MonthlyCharges` (r ≈ +0.19):** Weak positive correlation — higher bills slightly increase churn probability. The boxplot shows churned customers had a higher median MonthlyCharges. Likely linked to Fiber optic users who pay more and churn more.

* **`Churn` vs `SeniorCitizen` (r ≈ +0.15):** Weak positive — seniors churn somewhat more. Not a dominant predictor on its own.

### Non-Technical Findings

1. **Customers who left had much shorter tenure** on average. The boxplot shows that churned customers' median tenure is roughly half that of retained customers. This means ConnectIQ loses most of its customers in the first 1–2 years — the early relationship is the most fragile and most important to protect.

2. **Churned customers paid higher monthly bills on average.** This seems counterintuitive — why would higher-paying customers leave? It likely reflects the Fiber optic segment: faster internet is more expensive, and those customers may feel the quality or value doesn't justify the price, or they can easily find a comparable deal elsewhere.

---
# F. Outliers

## F — Outlier Detection via IQR Boxplots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, col in enumerate(num_features):
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    lower  = q1 - 1.5 * iqr
    upper  = q3 + 1.5 * iqr
    n_out  = ((df[col] < lower) | (df[col] > upper)).sum()

    sns.boxplot(data=df, y=col, ax=axes[i], color='lightblue',
                flierprops=dict(marker='o', color='red', alpha=0.4, markersize=4))
    axes[i].set_title(f'Boxplot: {col}', fontsize=12)
    axes[i].set_xlabel(f'IQR outliers: {n_out} ({n_out/len(df)*100:.1f}%)')
    print(f'{col}:')
    print(f'  Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}')
    print(f'  Lower fence={lower:.2f}, Upper fence={upper:.2f}')
    print(f'  Outliers detected: {n_out} ({n_out/len(df)*100:.2f}%)')
    print()

plt.suptitle('Outlier Detection via IQR Boxplots', fontsize=14)
plt.tight_layout()
plt.show()

## F — Discussion

* **`tenure`:** Bounded between 0 and 72 months by design (6-year maximum contract history in the dataset). The IQR method detects **no outliers**. The distribution is wide but expected — no action needed.

* **`MonthlyCharges`:** A small number of values fall above the upper IQR fence (~\$115+). These correspond to customers subscribed to **all available add-ons** simultaneously (Fiber optic + all streaming + all security/backup services). These are **legitimate, real business values** representing a premium customer segment — not data errors. → **Retain as-is**.

* **`TotalCharges`:** The right tail extends to ~\$8,000+ (long-tenure, high-bill customers). These are **genuine data points** — customers who have been with ConnectIQ for 5–6 years and have accumulated large totals. No errors or anomalies. → **Retain**, but apply **log-transformation** (`np.log1p`) to reduce the influence of the long tail before linear or distance-based models (or simply drop it as recommended in Section E due to multicollinearity).

---
# G. Transforming `tenure`

## G1 — Identify the Different Values in `tenure`

In [ ]:
tenure_vals = sorted(df['tenure'].unique())
print(f'Range             : {min(tenure_vals)} – {max(tenure_vals)} months')
print(f'Number of unique values: {len(tenure_vals)}')
print(f'All values        : {tenure_vals}')

## G2 — Categorize `tenure` into Three Groups: 0, 1, 2

In [ ]:
def categorize_tenure(t):
    if t <= 24:
        return 0   # Short-term  (0–24 months / 0–2 years)
    elif t <= 48:
        return 1   # Mid-term    (25–48 months / 2–4 years)
    else:
        return 2   # Long-term   (49–72 months / 4–6 years)

df['tenure_group'] = df['tenure'].apply(categorize_tenure)

print('Tenure group distribution:')
print(df['tenure_group'].value_counts().sort_index())
print()
print('Group legend:')
print('  0 = Short-term  (tenure  0–24 months  /  0–2 years)')
print('  1 = Mid-term    (tenure 25–48 months  /  2–4 years)')
print('  2 = Long-term   (tenure 49–72 months  /  4–6 years)')
print()

# Verify boundaries
for g in [0, 1, 2]:
    sub = df[df['tenure_group'] == g]['tenure']
    print(f'  Group {g}: min tenure={sub.min()}, max tenure={sub.max()}, n={len(sub)}')

# Bar plot of the tenure group distribution
group_counts = df['tenure_group'].value_counts().sort_index()
labels = ['0 — Short-term\n(0–24 mo)', '1 — Mid-term\n(25–48 mo)', '2 — Long-term\n(49–72 mo)']
colors = ['#ff9999', '#99c2ff', '#99ff99']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, group_counts.values, color=colors, edgecolor='grey', linewidth=1.2)
ax.set_title('Tenure Group Distribution (0 = Short, 1 = Mid, 2 = Long)', fontsize=13)
ax.set_ylabel('Customer Count')
for bar, count in zip(bars, group_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 30, str(count), ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## G3 — Categorization Logic Explained

The `tenure` feature spans **0 to 72 months** (6 years). The three groups are created by splitting this range into **three equal 24-month windows**:

| Group | Label | Tenure Range | Business Meaning |
|:---:|:---|:---:|:---|
| **0** | Short-term | 0–24 months (0–2 yrs) | New customers — highest churn risk |
| **1** | Mid-term   | 25–48 months (2–4 yrs) | Established — moderate risk |
| **2** | Long-term  | 49–72 months (4–6 yrs) | Loyal veterans — lowest churn risk |

**Why these boundaries?**

* **Equal-width split:** Dividing 72 months into three equal 24-month windows is the natural, symmetric partition of the full tenure range. It avoids arbitrary cut points and treats each life-stage equally.
* **Business alignment:** The groups correspond to meaningful customer lifecycle stages — new/trial (Group 0), established (Group 1), and loyal (Group 2). Churn risk decreases monotonically across these groups, making the encoding interpretable and ordinal.
* **Bimodal histogram support:** The histogram from Section D shows spikes at both ends of the tenure range. This bimodal shape confirms that customers tend to land at the extremes: they either leave early (Group 0) or stay for years (Group 2), making a 3-group split more natural than a continuous feature.
* **Model simplicity:** An ordinal 3-level feature (`0`, `1`, `2`) is easier for tree-based models and logistic regression to exploit than 73 distinct integer values, and it reduces the influence of noise in the exact month count.

---
# H. Summary and Discussion

## H — Full EDA Summary, Findings, and Next Steps

---

### 1. Class Imbalance — Must Address Before Training

* `Churn` target: ~**73.5% No** vs ~**26.5% Yes** (~3:1 ratio) — significant **class imbalance**.
* A model trained on this raw distribution will be biased toward "No" and fail to identify most churners.
* **Rebalancing options:**
  * **SMOTE (Synthetic Minority Oversampling Technique):** generates synthetic samples for the minority class (churners). Preferred — adds information without discarding data.
  * **Random Undersampling:** removes majority-class rows at random. Simple but risks losing useful patterns.
  * **Class-weighted classifiers:** `class_weight='balanced'` in sklearn estimators penalizes minority misclassifications more — no resampling required.
* **Evaluation metric:** Use **F1-score**, **ROC-AUC**, or **Precision-Recall AUC** — NOT raw accuracy (misleading with imbalanced classes).

---

### 2. Columns to Drop Before Modelling

| Column | Reason |
|:---|:---|
| `customerID` | Unique row identifier — zero predictive value, maximum cardinality |
| `SupportTicketID` | Unique ticket reference — zero predictive value |
| `ActiveCountryCode` | **Zero-variance** constant (`+1`) — no information gain |
| `Education` | **80.6% missing** — imputation would manufacture the feature; too sparse to model |
| `TotalCharges` | **Multicollinear** with `tenure × MonthlyCharges` (r ≈ 0.83); redundant |

---

### 3. Missing Values — Resolved

* **`TotalCharges`** (11 rows, all `tenure == 0`): imputed with **0** — new customers not yet billed. ✅
* **`Education`** (~5,679 rows, ~80.6% missing): **column dropped** after Section C analysis. ✅

---

### 4. Distribution and Transformation Needs

* **`TotalCharges`** — **right-skewed** → apply `np.log1p()` transformation before linear/distance models if retained.
* **`MonthlyCharges`** and **`tenure`** — roughly symmetric; apply **StandardScaler** or **MinMaxScaler** for scale-sensitive models (SVM, KNN, logistic regression, neural networks).
* **Nominal categoricals** (`InternetService`, `PaymentMethod`, `Contract`) → **one-hot encoding**.
* **Binary categoricals** (Yes/No columns: `Partner`, `Dependents`, `PhoneService`, etc.) → binary 0/1 encoding.

---

### 5. Key Predictors Identified from EDA

Ranked roughly by signal strength observed in the plots:

1. **`Contract`** — Month-to-month customers churn at dramatically higher rates. Strongest categorical predictor found.
2. **`tenure`** / **`tenure_group`** — Short-tenure customers are at highest risk. Strong negative correlation with churn (r ≈ −0.35).
3. **`InternetService`** — Fiber optic users churn significantly more than DSL or no-internet customers.
4. **`PaymentMethod`** — Electronic check users churn far more than automatic payment users.
5. **`MonthlyCharges`** — Higher bills slightly increase churn probability (r ≈ +0.19).
6. **`SeniorCitizen`** — Seniors churn more despite being a small segment (~16%).
7. **`PaperlessBilling`** — Paperless billing users churn more.
8. **`OnlineSecurity` / `TechSupport`** — Customers *without* these add-ons churn at higher rates.

---

### 6. Feature Engineering

* **`tenure_group`** — Created in Section G: ordinal grouping (0 = short-term 0–24 mo, 1 = mid-term 25–48 mo, 2 = long-term 49–72 mo). Simpler and business-aligned alternative to raw continuous tenure.
* Potential future interaction features: `Contract × MonthlyCharges`, `InternetService × TechSupport`.

---

### 7. Multicollinearity / Redundant Feature Groups

* The six add-on service columns (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) all have a "No internet service" category directly determined by `InternetService`. After one-hot encoding, compute **VIF (Variance Inflation Factor)** and prune columns with VIF > 10 in linear models.
* `TotalCharges` is collinear with `tenure × MonthlyCharges` (r ≈ 0.83) — **drop it**.

---

### 8. Outliers

* No outliers were identified as data entry errors. High values in `MonthlyCharges` (premium subscribers) and `TotalCharges` (long-loyal customers) are legitimate. **No rows need removal** due to outliers.

---

### 9. Records to Consider Removing

* The **11 rows with `tenure == 0`** (brand-new customers) have no billing history (`TotalCharges = 0`) and may represent incomplete or noisy records. Consider removing them before training to avoid confusing the model, or retaining them since they do carry valid churn labels.

---

### Summary Action Table

| Action | Target | Reason |
|:---|:---|:---|
| **Drop columns** | `customerID`, `SupportTicketID`, `ActiveCountryCode`, `Education`, `TotalCharges` | ID / constant / sparse / multicollinear |
| **Fill NaN** | `TotalCharges` → 0 | New customers (tenure=0), logically correct |
| **Log-transform** | `TotalCharges` (if kept) | Right-skewed; reduces long-tail influence |
| **Scale** | `tenure`, `MonthlyCharges` | Required for distance/linear models |
| **One-hot encode** | `InternetService`, `PaymentMethod`, `Contract` | Nominal multi-class features |
| **Binary encode** | All Yes/No columns | Convert to 0/1 |
| **New feature** | `tenure_group` | Ordinal lifecycle grouping (0/1/2) |
| **Rebalance data** | Training set only | SMOTE or `class_weight='balanced'` |
| **Evaluation metric** | F1-score, ROC-AUC | Imbalanced class — don't use raw accuracy |
| **Check VIF** | Add-on service columns after encoding | Detect residual multicollinearity |